In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc
from tqdm import tqdm

from utils import set_seed
from train_loo import preprocess_adata
from profiler import profile_training

PROFILER_CSV_PATH = "../results/scaling_stats.csv"

In [ ]:
set_seed(0)

In [ ]:
slide_id = 'crc_242' 
dataset_name = slide_id

In [ ]:
import os
DATA_ROOT = os.environ.get("DATA_ROOT", ".")
dataset_path = os.path.join(DATA_ROOT, f"datasets/crc/raw_zenodo/{slide_id}.h5ad")
adata = sc.read_h5ad(dataset_path)

In [ ]:
adata = preprocess_adata(adata)

In [ ]:
from train_loo import DEFAULT_LABELS_KEY, DEFAULT_BATCH_KEY, DEFAULT_DOMAINS_KEY
batch_key = DEFAULT_BATCH_KEY
labels_key = DEFAULT_LABELS_KEY
domains_key = DEFAULT_DOMAINS_KEY

In [ ]:
adata.obs['celltype'] = adata.obs[labels_key]
adata.obs['mouse_id'] = str(slide_id)
adata.obs['region']   = adata.obs[domains_key]

In [ ]:
MODELS = [
    #'cellina',
    #'cpa', # only run in cpa_cuda env
    #'scgen',
    #'scvi',
    #'scanvi',
    #'scviva',  # always run scanvi before this 
    #'simvi',
    'cellina-graph',
    #'mintflow',
    #'spatialprop'
]

In [ ]:
sizes = [1e3, 1e4, 1e5]
datasets = [adata[np.random.choice(adata.shape[0], int(s), replace=False)].copy() for s in sizes]

In [ ]:
from sklearn.model_selection import train_test_split

def get_splits(adata, test_fraction=0.1):
    n_cells = adata.n_obs
    n_holdout = int(n_cells * test_fraction)

    # Randomly choose cells
    test_idx = np.random.choice(n_cells, n_holdout, replace=False)
    all_idx = np.arange(adata.n_obs)
    trainval_idx = np.setdiff1d(all_idx, test_idx)
    validation_size = 0.1
    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=validation_size,
        random_state=0,
        shuffle=False
    )
    return train_idx, val_idx, test_idx

In [ ]:
def slice_adata_by_distance(adata, n_cells=1000):
    # Your coordinate columns
    x_col = 'CenterX_global_px'
    y_col = 'CenterY_global_px'

    # Option 1: pick a center (e.g., tissue center)
    x_center = adata.obs[x_col].median()
    y_center = adata.obs[y_col].median()

    # Compute distance to center
    adata.obs['dist_to_center'] = np.sqrt(
        (adata.obs[x_col] - x_center)**2 + 
        (adata.obs[y_col] - y_center)**2
    )

    # Sort by distance and take top n_cells
    selected_cells = adata.obs.sort_values('dist_to_center').iloc[:n_cells].index

    # Slice the adata
    adata_slice = adata[selected_cells].copy()

    # Put spatial coords
    adata_slice.obsm['spatial'] = adata_slice.obs[['CenterX_global_px', 'CenterY_global_px']].values

    # Optional: drop temporary column
    adata_slice.obs.drop(columns=['dist_to_center'], inplace=True)

    import squidpy as sq
    sq.gr.spatial_neighbors(adata_slice)

    return adata_slice

In [ ]:
# Only run for mintflow and spatialprop
temp_adata_base_path = os.path.join(DATA_ROOT, "tmp")

# Save datasets to disk for mintflow, which only loads from disk
if 'mintflow' in MODELS or 'spatialprop' in MODELS:
    for s in sizes:
        dataset_path = f"{temp_adata_base_path}/{slide_id}_{int(s)}.h5ad"
        dataset = slice_adata_by_distance(adata, n_cells=int(s))
        dataset.write_h5ad(dataset_path)

In [ ]:
batch_size = 2048
max_epochs = 50

In [ ]:
for s, dataset in zip(sizes, datasets):
    splits = get_splits(dataset)
    for model_name in MODELS:
        print(f"Training {model_name}")
        if model_name == 'cellina':
            from cellina import CellinaModel
            from configs.cellina_config import MODEL_ARGS, TRAIN_ARGS, PLAN_KWARGS
            CellinaModel.setup_anndata(dataset, 
                                    batch_key=batch_key, 
                                    labels_key=labels_key, 
                                    domains_key=domains_key, 
                                    spatial_obsm_key='spatial_x', 
                                    layer='counts')
            model = CellinaModel(dataset, **MODEL_ARGS)

            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            
            profile_training(
                lambda: model.train(**TRAIN_ARGS, plan_kwargs=PLAN_KWARGS),
                model_name="cellina",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'cellina-graph':
            from cellina_graph import CellinaModel
            from configs.cellina_graph_config import MODEL_ARGS, TRAIN_ARGS, PLAN_KWARGS
            CellinaModel.setup_anndata(dataset, 
                                    batch_key=batch_key, 
                                    labels_key=labels_key, 
                                    domains_key=domains_key, 
                                    layer='counts',
                                    spatial_connectivities_key='spatial_connectivities', 
                                    )
            model = CellinaModel(dataset, **MODEL_ARGS)
            # Add split info
            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            
            profile_training(
                lambda: model.train(**TRAIN_ARGS, plan_kwargs=PLAN_KWARGS),
                model_name="cellina-graph",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'cpa':
            import cpa
            from configs.cpa_config import MODEL_ARGS, TRAIN_ARGS, PLAN_KWARGS
            dataset.obs['dose'] = 1.0 # NOTE: dummy dose for compatibility with CPA model
            dataset.obs['data_split'] = 'train'
            dataset.obs.iloc[splits[1], dataset.obs.columns.get_loc('data_split')] = 'valid'
            dataset.obs.iloc[splits[2], dataset.obs.columns.get_loc('data_split')] = 'test'
            cpa.CPA.setup_anndata(dataset,
                    perturbation_key=domains_key,
                    control_group='REF',
                    dosage_key='dose',
                    categorical_covariate_keys=[labels_key],
                    is_count_data=True,
                    max_comb_len=1,
                    )
            model = cpa.CPA(dataset,
                            split_key='data_split',
                            train_split='train',
                            valid_split='valid',
                            test_split='test',
                            **MODEL_ARGS)
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            
            profile_training(
                lambda: model.train(**TRAIN_ARGS, plan_kwargs=PLAN_KWARGS, save_path='cpa/'),
                model_name="cpa",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'cellina_graph':
            from cellina_graph import CellinaModel
            from configs.cellina_graph_config import MODEL_ARGS, TRAIN_ARGS, PLAN_KWARGS
            CellinaModel.setup_anndata(dataset, 
                                    batch_key=batch_key, 
                                    labels_key=labels_key, 
                                    domains_key=domains_key, 
                                    layer='counts',
                                    spatial_connectivities_key='spatial_connectivities', 
                                    )
            model = CellinaModel(dataset, **MODEL_ARGS)
            # Add split info
            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            
            profile_training(
                lambda: model.train(**TRAIN_ARGS, plan_kwargs=PLAN_KWARGS),
                model_name="cellina-graph",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'scgen':
            import pertpy as pt
            from configs.scgen_config import MODEL_ARGS, TRAIN_ARGS, PLAN_KWARGS
            sc.pp.normalize_total(dataset, target_sum=1e4)
            sc.pp.log1p(dataset)
            pt.tl.Scgen.setup_anndata(dataset, batch_key=domains_key, labels_key=labels_key)
            model = pt.tl.Scgen(dataset, **MODEL_ARGS)
            # Add split info
            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            
            profile_training(
                lambda: model.train(**TRAIN_ARGS, plan_kwargs=PLAN_KWARGS),
                model_name="scgen",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'scvi':
            from scvi.model import SCVI
            from configs.cellina_config import TRAIN_ARGS
            SCVI.setup_anndata(dataset, layer="counts", batch_key=batch_key)
            scvi_model = SCVI(dataset, gene_likelihood="nb", 
                              n_layers=2, n_latent=64)
            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            profile_training(
                lambda: scvi_model.train(**TRAIN_ARGS),
                model_name="scvi",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )
        
        if model_name == 'scanvi':
            from scvi.model import SCANVI
            from configs.cellina_config import TRAIN_ARGS
            SCANVI.setup_anndata(
                dataset,
                layer="counts",
                batch_key=batch_key,
                unlabeled_category="ignore",
                labels_key=labels_key,
            )
            scanvi_model = SCANVI(
                adata=dataset,
                n_layers=2,
                n_latent=64,
                linear_classifier=True,
            )
            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            
            profile_training(
                lambda: scanvi_model.train(**TRAIN_ARGS),
                model_name="scanvi",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'scviva':
            import scvi
            from configs.cellina_config import TRAIN_ARGS
            dataset.obsm["SCANVI"] = scanvi_model.get_latent_representation(adata=dataset,
                                                                            batch_size=batch_size)
            setup_kwargs = {
                "sample_key": batch_key,
                "labels_key": labels_key,
                "cell_coordinates_key": "spatial",
                # NOTE: according to their results, SCANVI embs worked better than scvi
                "expression_embedding_key": "SCANVI",
            }
            scvi.external.SCVIVA.preprocessing_anndata(
                dataset,
                k_nn=20,
                **setup_kwargs,
            )
            scvi.external.SCVIVA.setup_anndata(
                dataset,
                layer="counts",
                batch_key=batch_key,
                **setup_kwargs,
            )
            nichevae = scvi.external.SCVIVA(dataset, n_latent=64)
            TRAIN_ARGS['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            TRAIN_ARGS['batch_size'] = batch_size
            TRAIN_ARGS['max_epochs'] = max_epochs
            profile_training(
                lambda: nichevae.train(**TRAIN_ARGS),
                model_name="scviva",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'simvi':
            from simvi.model import SimVI
            from pytorch_lightning.utilities.seed import seed_everything
            seed_everything(0)
            
            SimVI.setup_anndata(dataset, batch_key=batch_key)
            n_neighbors = 50
            edge_index = SimVI.extract_edge_index(dataset, 
                                                  n_neighbors=n_neighbors, 
                                                  batch_key=batch_key)
            model = SimVI(dataset,
                          kl_weight=1,
                          kl_gatweight=0.01,
                          lam_mi=1000,
                          permutation_rate=0.5,
                          n_spatial=64,
                          n_intrinsic=64)
            
            profile_training(
                lambda: model.train(edge_index,
                        max_epochs=max_epochs,
                        batch_size=batch_size,
                        use_gpu=True,
                        mae_epochs=max_epochs,
                        device='cuda:0'),
                model_name="simvi",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

        if model_name == 'spatialprop':
            import torch
            from spatial_gnn.api.perturbation_api import train_perturbation_model
            exp_name = 'spatialprop_scaling'
            training_args = {
                'dataset':             dataset_name,
                'file_path':           f"{temp_adata_base_path}/{slide_id}_{int(s)}.h5ad",
                'train_ids':           [str(slide_id)],
                'test_ids':            [str(slide_id)],   # same — in-sample only
                'exp_name':            exp_name,
                'k_hop':               2,
                'augment_hop':         2,
                'center_celltypes':    'all',
                'node_feature':        'expression',
                'inject_feature':      'none',
                'learning_rate':       1e-3,
                'loss':                'weightedl1',
                'epochs':              max_epochs,
                'normalize_total':     True,
                'num_cells_per_ct_id': 100,
                'predict_celltype':    False,
                'pool':                'center',
                'do_eval':             False,
                'device':              'cuda' if torch.cuda.is_available() else 'cpu',
            }

            profile_training(
                lambda: train_perturbation_model(**training_args),
                model_name="spatialprop",
                num_epochs=max_epochs,
                dataset_name=dataset_name,
                dataset_size=dataset.n_obs,
                dataset_path=dataset_path,
                csv_path=PROFILER_CSV_PATH
            )

### Mintflow

Mintflow loads adata from disk only, so first save datasets in a temp folder, run mintflow, and delete datasets

In [ ]:
# Mintflow takes too long - reduce epochs and then scale to match other methods
max_epochs = 5

In [ ]:
if 'mintflow' in MODELS:
    for s in tqdm(sizes, desc="Mintflow train"):
        import mintflow
        num_epochs = max_epochs
        batch_size = batch_size
        labels_key = labels_key
        patient_id = batch_key
        n_neighbors = 10
        x_pos = 'CenterX_global_px'
        y_pos = 'CenterY_global_px'
        use_wandb = 'False'

        # Set up configs
        config_data_train, config_data_evaluation, config_model, config_training = \
            mintflow.get_default_configurations(
                num_tissue_sections_training=1,
                num_tissue_sections_evaluation=1
            )

        # training config
        config_data_train['list_tissue']['anndata1']['file'] = f"{temp_adata_base_path}/{slide_id}_{int(s)}.h5ad"
        config_data_train['list_tissue']['anndata1']['obskey_cell_type'] = labels_key
        config_data_train['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = patient_id
        config_data_train['list_tissue']['anndata1']['obskey_x'] = x_pos
        config_data_train['list_tissue']['anndata1']['obskey_y'] = y_pos
        config_data_train['list_tissue']['anndata1']['obskey_biological_batch_key'] = patient_id
        config_data_train['list_tissue']['anndata1']['config_dataloader_train']['width_window'] = batch_size
        config_data_train['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
            'n_neighs': n_neighbors,
            'set_diag': 'False',
            'delaunay': 'False',
        }

        config_data_evaluation['list_tissue']['anndata1']['file'] = f"{temp_adata_base_path}/{slide_id}_{int(s)}.h5ad"
        config_data_evaluation['list_tissue']['anndata1']['obskey_cell_type'] = labels_key
        config_data_evaluation['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = patient_id
        config_data_evaluation['list_tissue']['anndata1']['obskey_x'] = x_pos
        config_data_evaluation['list_tissue']['anndata1']['obskey_y'] = y_pos
        config_data_evaluation['list_tissue']['anndata1']['obskey_biological_batch_key'] = patient_id
        config_data_evaluation['list_tissue']['anndata1']['config_dataloader_test']['width_window'] = batch_size
        config_data_evaluation['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
            'n_neighs': n_neighbors,
            'set_diag': 'False',
            'delaunay': 'False',
        }

        config_data_train = mintflow.verify_and_postprocess_config_data_train(config_data_train) 
        config_data_evaluation = mintflow.verify_and_postprocess_config_data_evaluation(config_data_evaluation)

        config_model = mintflow.verify_and_postprocess_config_model(
            config_model,
            num_tissue_sections=len(config_data_train)
        )

        config_training['num_training_epochs'] = num_epochs
        config_training['flag_enable_wandb'] = use_wandb
        config_training['flag_finaleval_createanndata_alltissuescombined'] = 'True'
        config_training = mintflow.verify_and_postprocess_config_training(config_training)

        dict_all4_configs = {
            "config_data_train": config_data_train,
            "config_data_evaluation": config_data_evaluation,
            "config_model": config_model,
            "config_training": config_training,
        }

        data_mintflow = mintflow.setup_data(dict_all4_configs=dict_all4_configs)

        model = mintflow.setup_model(
            dict_all4_configs=dict_all4_configs,
            data_mintflow=data_mintflow
        )

        trainer = mintflow.Trainer(
            dict_all4_configs=dict_all4_configs,
            model=model,
            data_mintflow=data_mintflow
        )

        def mintflow_train_loop(trainer, dict_all4_configs):
            for epoch in tqdm(range(dict_all4_configs["config_training"]["num_training_epochs"]), desc="Training Epochs"):
                trainer.train_one_epoch()
                
        profile_training(
            lambda: mintflow_train_loop(trainer, dict_all4_configs),
            model_name="mintflow",
            num_epochs=num_epochs,
            dataset_name=dataset_name,
            dataset_size=int(s),
            dataset_path=f"{temp_adata_base_path}/{slide_id}_{int(s)}.h5ad",
            csv_path=PROFILER_CSV_PATH
        )

# Plotting

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import matplotlib as mpl
mpl.rcParams["font.family"] = "monospace"

In [ ]:
df = pd.read_csv(PROFILER_CSV_PATH)

In [ ]:
# for model_name == 'scviva' for a certain dataset_name + dataset_size combination, 
# add the 'scanvi' total train time to the 'scviva' total train time 
# to account for the fact that scanvi is a required preprocessing step for scviva
for dname in df['dataset_name'].unique():
    for dataset_size in df['dataset_size'].unique():
        scanvi_time = df[(df['model_name'] == 'scanvi') & (df['dataset_name'] == dname) & (df['dataset_size'] == dataset_size)]['total_train_time_sec'].sum()
        df.loc[(df['model_name'] == 'scviva') & (df['dataset_name'] == dname) & (df['dataset_size'] == dataset_size), 'total_train_time_sec'] += scanvi_time

In [ ]:
# for mintflow, multiple total train time by 10 to account for the fact that we reduced epochs from 50 to 5
df.loc[df['model_name'] == 'mintflow', 'total_train_time_sec'] *= 10

In [ ]:
palette_dict = {
    # HEX code for cellina purple: #8172B2
    'cellina': '#8172B2',
    'scgen':   'orange',
    'scvi':    'green',
    'scanvi':  'red',
    'cpa':     'teal',
    'scviva':  'brown',
    'simvi':   'grey',
    'mintflow': 'pink',
    'spatialprop': 'skyblue',
    'cellina-graph': 'purple',
}
palette = palette_dict

In [ ]:
# Define which models are spatial
spatial_models = ['cellina', 'scviva', 'simvi', 'mintflow', 'spatialprop', 'cellina-graph']
# Non-spatial
non_spatial_models = ['scgen', 'scvi', 'scanvi', 'cpa']

In [ ]:
df_sub = df[df['dataset_name'] == dataset_name]

In [ ]:
df_sub = df_sub.groupby(['model_name', 'dataset_size'], as_index=False)['total_train_time_sec'].mean()
# Add a column to df_sub indicating line style
df_sub['type'] = df_sub['model_name'].apply(
    lambda x: 'spatial' if x in spatial_models else 'non-spatial'
)

In [ ]:
plt.figure(figsize=(8,5))
sns.lineplot(
    data=df_sub,
    x='dataset_size',
    y='total_train_time_sec',
    hue='model_name',
    marker='o',
    palette=palette,
    style='type',       # use this column for solid/dashed
    markers=True,
    dashes={'spatial': '', 'non-spatial': (5,5)},  # optional: specify dash pattern
)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of cells')
plt.ylabel('Total training time (sec)')
plt.title(f'Model scaling with dataset size ({dataset_name})')

leg = plt.legend(
    bbox_to_anchor=(1.05, 1),  # x=1.05 moves it slightly outside to the right
    loc='upper left',
    borderaxespad=0,
)

# Bold only category headings
category_headings = ['model_name', 'type']  # adjust to your legend category names
for text in leg.get_texts():
    if text.get_text() in category_headings:
        text.set_fontweight('bold')

plt.tight_layout()
plt.savefig(f'../figures/model_scaling_{dataset_name}.svg', bbox_inches="tight")
plt.show()

In [ ]:
df_avg = df.groupby(['model_name', 'dataset_size'], as_index=False)['total_train_time_sec'].mean()
# Add a column to df_avg indicating line style
df_avg['type'] = df_avg['model_name'].apply(
    lambda x: 'spatial' if x in spatial_models else 'non-spatial'
)

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=df_avg,
    x='dataset_size',
    y='total_train_time_sec',
    hue='model_name',
    marker='o',
    palette=palette,
    style='type',       # use this column for solid/dashed
    markers=True,
    dashes={'spatial': '', 'non-spatial': (5,5)},  # optional: specify dash pattern
    #legend='full'
)
plt.xscale('log')  # optional: for wide range of dataset sizes
plt.yscale('log')  # optional: for wide range of times
plt.xlabel('Number of cells', fontsize=14)
plt.ylabel('Average training time (sec)', fontsize=14)
plt.title(f'Model scaling (mean over slides)', fontsize=18)

leg = plt.legend(
    bbox_to_anchor=(1.05, 1),  # x=1.05 moves it slightly outside to the right
    loc='upper left',
    borderaxespad=0,
)

# Bold only category headings
category_headings = ['model_name', 'type']  # adjust to your legend category names
for text in leg.get_texts():
    if text.get_text() in category_headings:
        text.set_fontweight('bold')
        text.set_fontsize(12)
    else:
        text.set_fontsize(12)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.savefig(f'../figures/model_scaling_average.svg', format='svg', bbox_inches="tight")
plt.show()